# 🇭🇷 VEPRAD Croatian Speech — CTC Segmentation Pipeline

This notebook aligns VEPRAD audio recordings with their transcripts and splits them into short segments ready for ASR training using NVIDIA NeMo.

---

### 📁 Expected folder structure
```
Croatian-Language-Ctc_Segmentation/
├── VEPRAD/
│   ├── audio_m/      ← male audio
│   ├── audio_z/      ← female audio
│   └── transkripcije_m+z/   ← transcripts
└── veprad_ctc_segmentation.ipynb  ← this notebook
```

In [ ]:
# Install required packages
# This takes a few minutes to complete
%pip install nemo_toolkit["all"]
%pip install ctc-segmentation

In [ ]:
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
VEPRAD_ROOT      = "./VEPRAD"                          # root of your VEPRAD folder
AUDIO_DIRS       = [
    os.path.join(VEPRAD_ROOT, "audio_m"),       # male speakers
    os.path.join(VEPRAD_ROOT, "audio_z"),       # female speakers
]
TRANSCRIPT_DIR   = os.path.join(VEPRAD_ROOT, "transkripcije_m+z")
CLEAN_DIR        = "./veprad_transcripts_clean"        # cleaned transcripts output
OUTPUT_DIR       = "./veprad_segmented"                # segmented wav files output
MANIFEST_PATH    = "./veprad_manifest.json"            # NeMo manifest output

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME       = "nvidia/stt_hr_conformer_ctc_large" # Croatian CTC model
SAMPLE_RATE      = 16000

# ── Segmentation settings ──────────────────────────────────────────────────
# THRESHOLD: alignment confidence cutoff.
#   -10  = strict  (only keep well-aligned segments)
#   -20  = lenient (keep more segments, some may be noisy)
THRESHOLD        = -10

# WINDOW_LEN: how many tokens to consider at once.
#   Increase if your audio files are very long (> 5 minutes)
WINDOW_LEN       = 8000

# ── Verify paths exist ─────────────────────────────────────────────────────
for d in [VEPRAD_ROOT, TRANSCRIPT_DIR] + AUDIO_DIRS:
    status = "✅" if Path(d).exists() else "❌ NOT FOUND"
    print(f"{status}  {d}")

In [ ]:
transcript_files = sorted(Path(TRANSCRIPT_DIR).rglob("*.txt"))

print(f"📂 Total transcript files found: {len(transcript_files)}")
print("\n" + "─"*100)
print("📄 Previewing first 5 transcript files (raw):")
print("─"*100)

for f in transcript_files[200:205]:
    content = f.read_text(encoding="utf-8", errors="replace")
    print(f"\n▶ {f.name}")
    print(content[:400])  # show first 300 chars
    print("─"*100)

---
## 🔧 Cell 4 — Encoding Fix Map
Based on what you saw above, verify or adjust the symbol mapping below.

| Symbol in file | Croatian character |
|---|---|
| `{` | `š` |
| `^` | `ć` |
| `~` | `č` |
| ` | `ž` |
| `\\` | `đ` |

**If any of these are wrong, edit the `ENCODING_MAP` dict before continuing.**

In [ ]:
import re

ENCODING_MAP = {
    '{' : 'š',
    '^' : 'ć',
    '~' : 'č',
    '`' : 'ž',
}

def fix_encoding(text: str) -> str:
    """Replace VEPRAD encoding artifacts with correct Croatian characters."""
    for old, new in ENCODING_MAP.items():
        text = text.replace(old, new)
    return text

def clean_transcript(text: str) -> str:
    """Remove non-speech annotations and fix encoding."""
    text = fix_encoding(text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\[[^\]]+\]", "", text)
    text = text.lower()
    text = re.sub(r"[^a-zčćđšž\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text